# 02 - Extract JRC Production Route Data

## Purpose
Extract the hydrogen production route emission intensities from JRC135067.
This dataset provides production route granularity not available in the
CBAM default values xlsx.

## Inputs
- `data/raw/JRC135067_01.pdf` - Hydrogen (global average methodology)

## Output
- `data/processed/hydrogen_route_intensities.csv`

## Notes
- JRC134682 Table 1 was assessed and determined redundant. The 
  production_route column in cbam_defaults.csv encodes production route 
  and steel grade directly via letter codes defined in the annex to 
  Commission Implementing Regulation (EU) 2025/2620. These map onto 
  BF/BOF, DRI/EAF and Scrap/EAF routes, covering the same information as 
  Table 1 at sufficient granularity for this analysis. Not extracted.
- JRC135067 Table 2 contains global average emission intensities per
  hydrogen production route. 6 rows, manually verified against source.
- Annex 2 of JRC134682 (country-level emission intensities) was assessed
  and determined to be redundant with the CBAM default values xlsx, which
  covers 119 countries vs the annex's 15-20, with the same direct/indirect/
  total columns. Not extracted.

## JRC135067 Table 2 - Hydrogen Production Route Emission Intensities

Table 2 provides global average GHG emission intensities per hydrogen
production route. Unlike JRC134682 which provides country-level data,
this report estimates a single global average figure per route, reflecting
the current global production mix (based on 2021 IEA data).

### Structure Notes
- Source: JRC135067, internal page 6, PDF page 9
- 4 columns: feedstock_type, total_emissions_tco2_per_th2, comments, source
- 6 data rows covering: natural gas, coal, naphtha, oil,
  electrolysis chlor-alkali, electrolysis water
- No country breakdown. Global average only.
- Notable finding: water electrolysis at 23.1 tCO2/tH2 is higher than
  natural gas at 9.0, due to current global average grid emission intensity.
  This reverses as grids decarbonize.

In [ ]:
import pandas as pd
from pathlib import Path
import pdfplumber

pdf_path = Path.cwd().parent / "data" / "raw" / "JRC135067_01.pdf"

# Table 2 is on internal page 6, PDF page 9 (0-indexed: page 8)
# "GHG emission intensities associated with the different hydrogen production routes"

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[8]
    tables = page.extract_tables()
    print(f"Tables found on page 9: {len(tables)}")
    for i, t in enumerate(tables):
        print(f"\nTable {i}: {len(t)} rows x {len(t[0])} cols")
        for row in t:
            print(f"  {row}")

Tables found on page 9: 1

Table 0: 6 rows x 2 cols
  ['9', '-']
  ['19.2', '-']
  ['7', 'This value considers the use of natural gas\ncovering the heat requirements of the process\nand substituting for exported hydrogen.']
  ['12', '-']
  ['7', 'This value considers the use of natural gas\ncovering the heat requirements of the process\nand substituting for exported hydrogen.']
  ['23.1', 'Despite the significant amount of total\nemissions associated to this pathway,\nelectrolysis has a negligible impact on the\nglobal average emission value because of its\nsmall global volumes.']


### Extraction Attempt
Text extraction was attempted using pdfplumber. The table structure in
JRC135067 page 9 causes pdfplumber to read across columns left to right,
mixing feedstock names, values and comments into a single text stream.
Table extraction returned only 2 of 4 columns with no header row.
Given the table has 6 rows with fully verifiable values, the DataFrame
is constructed directly from the extracted text output, with all values
cross-checked against the source document.

In [2]:
# Step 1: Extract raw text from page to document what pdfplumber returns
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[8]
    text = page.extract_text()

start_marker = "Table 2. GHG emission intensities associated with the different hydrogen production routes"
end_marker = "Source: As specified in the Table"

start_idx = text.find(start_marker)
end_idx = text.find(end_marker)

table_text = text[start_idx:end_idx].strip()
print("=== Raw extracted text ===")
print(table_text)

# Step 2: pdfplumber cannot reliably parse this table due to multi-column
# PDF rendering mixing feedstock names, values and comments in a single
# text stream. DataFrame constructed directly from verified source values.
# All figures cross-checked against JRC135067 internal page 6.

hydrogen_routes = pd.DataFrame([
    {"feedstock_type": "Natural gas",
     "total_emissions_tco2_per_th2": 9.0,
     "comments": "",
     "source": "IEA 2023"},
    {"feedstock_type": "Coal",
     "total_emissions_tco2_per_th2": 19.2,
     "comments": "",
     "source": "IEA 2023"},
    {"feedstock_type": "Naphtha (by-product)",
     "total_emissions_tco2_per_th2": 7.0,
     "comments": "Value considers natural gas covering heat requirements, substituting for exported hydrogen.",
     "source": "Lee & Elgowainy 2018"},
    {"feedstock_type": "Oil",
     "total_emissions_tco2_per_th2": 12.0,
     "comments": "",
     "source": "IEA 2019"},
    {"feedstock_type": "Electrolysis (chlor-alkali)",
     "total_emissions_tco2_per_th2": 7.0,
     "comments": "Value considers natural gas covering heat requirements, substituting for exported hydrogen.",
     "source": "Lee et al. 2018"},
    {"feedstock_type": "Electrolysis (water, world average)",
     "total_emissions_tco2_per_th2": 23.1,
     "comments": "High emissions due to current global grid mix. Negligible impact on global average given small production volumes.",
     "source": "IEA 2023"},
])

print("\n=== Constructed DataFrame ===")
print(hydrogen_routes.to_string())

=== Raw extracted text ===
Table 2. GHG emission intensities associated with the different hydrogen production routes
Total
emissions/ Comments
Feedstock
type tCO /tH Source
2 2
Natural Gas 9 - [3]
Coal 19.2 - [3]
This value considers the use of natural gas
7 covering the heat requirements of the process
Naphtha and substituting for exported hydrogen. [4]
Oil 12 - [5]
This value considers the use of natural gas
Electrolysis 7 covering the heat requirements of the process
(chlor-alkali) and substituting for exported hydrogen. [6]
Despite the significant amount of total
Electrolysis emissions associated to this pathway,
(water), 23.1 electrolysis has a negligible impact on the
world global average emission value because of its
average small global volumes. [3]

=== Constructed DataFrame ===
                        feedstock_type  total_emissions_tco2_per_th2                                                                                                            comments                

### CN Code for Hydrogen

Unlike steel and other CBAM materials which cover many distinct products
each with their own CN code, hydrogen has a single CN code under CBAM:
**2804 10 00**. 

Each CN code can be produced via multiple production routes,
but the code itself identifies the product, not the route.
Source: Annex I, Regulation (EU) 2023/956.
https://eur-lex.europa.eu/eli/reg/2023/956/oj/eng

CN code added as first column for consistency with other processed datasets
and to enable joins in the database layer.

In [3]:
# Add CN code as first column
hydrogen_routes.insert(0, "cn_code", "2804 10 00")
print(hydrogen_routes.to_string())

      cn_code                       feedstock_type  total_emissions_tco2_per_th2                                                                                                            comments                source
0  2804 10 00                          Natural gas                           9.0                                                                                                                                  IEA 2023
1  2804 10 00                                 Coal                          19.2                                                                                                                                  IEA 2023
2  2804 10 00                 Naphtha (by-product)                           7.0                         Value considers natural gas covering heat requirements, substituting for exported hydrogen.  Lee & Elgowainy 2018
3  2804 10 00                                  Oil                          12.0                                            

### Output
Saving cleaned Table 1 to `data/processed/hydrogen_route_intensities.csv`

In [ ]:
output_path = Path.cwd().parent / "data" / "processed" / "hydrogen_route_intensities.csv"
hydrogen_routes.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Final shape: {hydrogen_routes.shape}")

Saved to: /Users/milcahmaryJoseph/Documents/GitHub/cbam-analysis/data/processed/jrc_cn_hydrogen_routes.csv
Final shape: (6, 5)
